# Notebook 4: Subgroup & Ablation & Clinical Analysis
Input: Triage 코드 + Notebook1 Output (데이터) + Notebook3 Output (모든 체크포인트)

In [ ]:
# 1. GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU!')

In [ ]:
# 2. 코드 복사
import os, shutil, glob
from pathlib import Path

def find_input_dir():
    candidates = glob.glob('/kaggle/input/**/configs', recursive=True)
    if candidates:
        return str(Path(candidates[0]).parent)
    return None

input_dir = find_input_dir()
assert input_dir, 'Triage 데이터셋을 찾을 수 없습니다'
dest = '/kaggle/working/Triage'
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree(input_dir, dest)

# 새 분석 스크립트 복사 (데이터셋 루트에 업로드된 경우)
dataset_root = str(Path(input_dir).parent)
new_scripts = [
    'analyze_subgroups.py',
    'analyze_ablation.py',
    'analyze_missingness_viz.py',
    'analyze_clinical_comparison.py',
    'analyze_mc_dropout.py',
    'analyze_ewt.py',
    'analyze_imputation_comparison.py',
]
for script in new_scripts:
    src = os.path.join(dataset_root, script)
    if os.path.exists(src):
        shutil.copy2(src, f'{dest}/scripts/{script}')
        print(f'  복사: {script}')
print('코드 복사 완료')

In [ ]:
# 3. 데이터 + 체크포인트 복사
import os, shutil, zipfile, glob
from pathlib import Path

def extract_zip(zip_path, extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)

# Notebook1 output (처리된 데이터)
nb1_zip = '/kaggle/input/notebooks/parkyunhu1/notebook722c6297be/_output_.zip'
print('Notebook1 압축 해제 중...')
extract_zip(nb1_zip, '/kaggle/working/nb1_output')
print('완료')

candidates = glob.glob('/kaggle/working/nb1_output/**/processed', recursive=True)
assert candidates, 'processed 폴더 없음'
data_src = str(Path(candidates[0]).parent)

data_dest = '/kaggle/working/Triage/data'
if os.path.exists(data_dest):
    shutil.rmtree(data_dest)
shutil.copytree(data_src, data_dest)
print('데이터 복사 완료')

# Notebook3 output (모든 체크포인트) — best.pt만 선택적 추출
all_zips = glob.glob('/kaggle/input/notebooks/parkyunhu1/*/_output_.zip')
nb3_zips = [z for z in all_zips if 'notebook722c6297be' not in z]

os.makedirs('/kaggle/working/Triage/results', exist_ok=True)
for i, nb_zip in enumerate(nb3_zips):
    print(f'Notebook{i+2} best.pt 추출 중: {nb_zip}')
    with zipfile.ZipFile(nb_zip, 'r') as zf:
        best_files = [f for f in zf.namelist() if f.endswith('_best.pt')]
        for fname in best_files:
            target = Path('/kaggle/working/Triage/results') / Path(fname).relative_to(Path(fname).parts[0])
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(fname) as src, open(target, 'wb') as dst:
                dst.write(src.read())
    print(f'  {len(best_files)}개 체크포인트 추출')

ckpts = list(Path('/kaggle/working/Triage/results').glob('**/*_best.pt'))
print(f'\n총 체크포인트: {len(ckpts)}개')
for c in sorted(ckpts):
    print(' -', c.name)

In [ ]:
# 4. 패키지 설치
!pip install -q omegaconf lifelines statsmodels pyarrow scipy

In [ ]:
# 5. Subgroup 분석
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'

result = subprocess.run(
    [sys.executable, 'scripts/analyze_subgroups.py'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout[-8000:] if len(result.stdout) > 8000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# 6. Ablation 분석
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'

result = subprocess.run(
    [sys.executable, 'scripts/analyze_ablation.py'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# 7. Missingness 시각화 데이터 추출
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'

result = subprocess.run(
    [sys.executable, 'scripts/analyze_missingness_viz.py'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# 8. B1: Clinical Score 비교 (qSOFA / SOFA vs IMST-Mamba)
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'

result = subprocess.run(
    [sys.executable, 'scripts/analyze_clinical_comparison.py'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout[-6000:] if len(result.stdout) > 6000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# 9. B2: MC Dropout 불확실성 정량화
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'

result = subprocess.run(
    [sys.executable, 'scripts/analyze_mc_dropout.py'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout[-6000:] if len(result.stdout) > 6000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# 10. B3: Early Warning Time (EWT) 전체 모델 비교
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'

result = subprocess.run(
    [sys.executable, 'scripts/analyze_ewt.py'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout[-6000:] if len(result.stdout) > 6000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# 11. B4: Imputation 전략 비교 (Transformer vs IMST-Mamba)
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'

result = subprocess.run(
    [sys.executable, 'scripts/analyze_imputation_comparison.py'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout[-6000:] if len(result.stdout) > 6000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# 12. 전체 결과 출력 및 저장
import json, shutil, os

result_files = [
    'subgroup_analysis.json',
    'ablation_results.json',
    'missingness_viz.json',
    'clinical_comparison.json',
    'mc_dropout_results.json',
    'ewt_analysis.json',
    'imputation_comparison.json',
]

for fname in result_files:
    fpath = f'/kaggle/working/Triage/results/{fname}'
    if not os.path.exists(fpath):
        print(f'{fname} 없음')
        continue
    with open(fpath) as f:
        data = json.load(f)
    print(f'\n=== {fname} ===')

    if fname == 'subgroup_analysis.json':
        for model, sgs in data.items():
            print(f'  {model}:')
            for sg, r in sgs.items():
                if r and 'auroc' in r:
                    print(f'    {sg:<20} AUROC={r["auroc"]:.4f}  AUPRC={r["auprc"]:.4f}')

    elif fname == 'ablation_results.json':
        for abl, r in data.items():
            if r and 'auroc' in r:
                print(f'  {abl:<20} AUROC={r["auroc"]:.4f}  AUPRC={r["auprc"]:.4f}')

    elif fname == 'clinical_comparison.json':
        cm = data.get('clinical_metrics', {})
        ed = data.get('early_detection', {})
        print('  Clinical metrics:')
        for name, r in cm.items():
            if r and 'auroc' in r:
                print(f'    {name:<20} AUROC={r["auroc"]:.4f}  Sens={r.get("sensitivity",0):.4f}  Spec={r.get("specificity",0):.4f}')
        print('  Early detection time:')
        for name, e in ed.items():
            print(f'    {name:<20} mean EWT={e["mean_hours"]:.2f}h  early%={100*e["pct_early"]:.1f}%')

    elif fname == 'mc_dropout_results.json':
        mc = data.get('mc_metrics', {})
        ece = data.get('ece', float('nan'))
        print(f'  AUROC={mc.get("auroc","N/A")}  AUPRC={mc.get("auprc","N/A")}  ECE={ece:.4f}')

    elif fname == 'ewt_analysis.json':
        for name, r in data.get('ewt_summary', {}).items():
            if r:
                print(f'  {name:<20} mean={r.get("mean_ewt_h",float("nan")):.2f}h  '
                      f'early%={100*r.get("pct_early",0):.1f}%')

    elif fname == 'imputation_comparison.json':
        for name, grp in data.items():
            ov = grp.get('overall', {}).get('auroc', float('nan'))
            hi = grp.get('miss_high', {}).get('auroc', float('nan'))
            print(f'  {name:<35} overall={ov:.4f}  high_miss={hi:.4f}')

# Output 저장
if os.path.exists('/kaggle/working/results'):
    shutil.rmtree('/kaggle/working/results')
shutil.copytree('/kaggle/working/Triage/results', '/kaggle/working/results')
print('\nOutput 저장 완료')